# **1. Setup and Configuration**


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Define current date for calculations
CURRENT_DATE = datetime.now().date()
print(f"Analysis Date: {CURRENT_DATE}")



# **2. Load Data from Bronze Layer**

In [0]:


# Load all tables 
employee_df = spark.table("workspace.bronze.employee_data")
engagement_df = spark.table("workspace.bronze.employee_engagement_survey_data")
recruitment_df = spark.table("workspace.bronze.recruitment_data")
training_df = spark.table("workspace.bronze.training_and_development_data")

# **3. Initial Data Profiling**


## **3.1 Dataset Dimensions**


In [0]:
tables = {
    'employee_data': employee_df,
    'engagement_survey': engagement_df,
    'recruitment_data': recruitment_df,
    'training_data': training_df
}

print("="*60)
print("DATASET DIMENSIONS")
print("="*60)
for name, df in tables.items():
    row_count = df.count()
    col_count = len(df.columns)
    print(f"{name:.<30} {row_count:>10,} rows × {col_count:>3} columns")


## **3.2 Missing Values Analysis**


In [0]:
def analyze_missing_values(df, table_name):
    """Calculate missing value percentages for each column"""
    total_rows = df.count()
    
    missing_data = []
    for col_name in df.columns:
        # Get column data type
        col_type = dict(df.dtypes)[col_name]
        
        # For string columns, check for null and empty strings
        if col_type == 'string':
            null_count = df.filter(
                F.col(col_name).isNull() | 
                (F.trim(F.col(col_name)) == "")
            ).count()
        # For non-string columns, only check for null
        else:
            null_count = df.filter(F.col(col_name).isNull()).count()
        
        missing_pct = (null_count / total_rows) * 100
        missing_data.append({
            'column': col_name,
            'data_type': col_type,
            'missing_count': null_count,
            'missing_percentage': round(missing_pct, 2)
        })
    
    missing_df = pd.DataFrame(missing_data).sort_values('missing_percentage', ascending=False)
    
    print(f"\n{'='*60}")
    print(f"MISSING VALUES - {table_name.upper()}")
    print(f"{'='*60}")
    print(missing_df[missing_df['missing_percentage'] > 0].to_string(index=False))
    
    return missing_df

# Analyze each table
employee_missing = analyze_missing_values(employee_df, "employee_data")
engagement_missing = analyze_missing_values(engagement_df, "engagement_survey")
recruitment_missing = analyze_missing_values(recruitment_df, "recruitment_data")
training_missing = analyze_missing_values(training_df, "training_data")

# **4. Employee Data - Core Analysis**

## **4.1 Calculate Key Metrics (Tenure, Age, Attrition Flag)**

In [0]:
# Create enriched employee dataframe with calculated fields
employee_enriched = employee_df.withColumn(
    "ExitDate_parsed", 
    F.when(F.col("ExitDate").isNotNull() & (F.col("ExitDate") != ""), 
           F.to_date(F.col("ExitDate"), "dd-MMM-yy"))
).withColumn(
    "StartDate_parsed",
    F.to_date(F.col("StartDate"), "dd-MMM-yy")
).withColumn(
    "is_active",
    F.when(F.col("ExitDate_parsed").isNull(), 1).otherwise(0)
).withColumn(
    "tenure_days",
    F.datediff(
        F.coalesce(F.col("ExitDate_parsed"), F.lit(CURRENT_DATE)),
        F.col("StartDate_parsed")
    )
).withColumn(
    "tenure_months",
    (F.col("tenure_days") / 30.44).cast("integer")
).withColumn(
    "tenure_years",
    F.round(F.col("tenure_days") / 365.25, 1)
).withColumn(
    "age",
    F.floor(F.months_between(F.lit(CURRENT_DATE), F.col("DOB")) / 12)
).withColumn(
    "age_group",
    F.when(F.col("age") < 25, "Under 25")
     .when((F.col("age") >= 25) & (F.col("age") < 35), "25-34")
     .when((F.col("age") >= 35) & (F.col("age") < 45), "35-44")
     .when((F.col("age") >= 45) & (F.col("age") < 55), "45-54")
     .otherwise("55+")
).withColumn(
    "exit_year",
    F.year(F.col("ExitDate_parsed"))
).withColumn(
    "exit_month",
    F.month(F.col("ExitDate_parsed"))
)


print("✅ Created enriched employee dataset with calculated fields")
display(employee_enriched.select("EmpID", "FirstName", "LastName", "is_active", "tenure_years", "age", "age_group").limit(10))

## **4.2 Overall Attrition Metrics**

In [0]:
# Calculate key attrition statistics
total_employees = employee_enriched.count()
active_employees = employee_enriched.filter(F.col("is_active") == 1).count()
departed_employees = employee_enriched.filter(F.col("is_active") == 0).count()
attrition_rate = (departed_employees / total_employees) * 100

# Voluntary vs Involuntary
voluntary_count = employee_enriched.filter(
    (F.col("is_active") == 0) & 
    (F.lower(F.col("TerminationType")).contains("voluntary"))
).count()

involuntary_count = employee_enriched.filter(
    (F.col("is_active") == 0) & 
    (F.lower(F.col("TerminationType")).contains("involuntary"))
).count()

print("="*60)
print("OVERALL ATTRITION METRICS")
print("="*60)
print(f"Total Employees:           {total_employees:>10,}")
print(f"Active Employees:          {active_employees:>10,}")
print(f"Departed Employees:        {departed_employees:>10,}")
print(f"Overall Attrition Rate:    {attrition_rate:>10.2f}%")
print(f"\nVoluntary Departures:      {voluntary_count:>10,}")
print(f"Involuntary Departures:    {involuntary_count:>10,}")
if departed_employees > 0:
    print(f"Voluntary %:               {(voluntary_count/departed_employees)*100:>10.2f}%")

## **4.3 Attrition by Department**

In [0]:
dept_attrition = employee_enriched.groupBy("DepartmentType").agg(
    F.count("*").alias("total_employees"),
    F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)).alias("departed"),
    F.sum(F.when(F.col("is_active") == 1, 1).otherwise(0)).alias("active")
).withColumn(
    "attrition_rate",
    F.round((F.col("departed") / F.col("total_employees")) * 100, 2)
).orderBy(F.desc("attrition_rate"))

print("ATTRITION BY DEPARTMENT")
print("="*60)
display(dept_attrition)


# Visualize department attrition
dept_attrition_pd = dept_attrition.toPandas()

plt.figure(figsize=(12, 6))
plt.barh(dept_attrition_pd['DepartmentType'], dept_attrition_pd['attrition_rate'], color='coral')
plt.xlabel('Attrition Rate (%)', fontsize=12)
plt.ylabel('Department', fontsize=12)
plt.title('Attrition Rate by Department', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


## **4.4 Attrition by Business Unit**

In [0]:

bu_attrition = employee_enriched.groupBy("BusinessUnit").agg(
    F.count("*").alias("total_employees"),
    F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)).alias("departed"),
    F.sum(F.when(F.col("is_active") == 1, 1).otherwise(0)).alias("active")
).withColumn(
    "attrition_rate",
    F.round((F.col("departed") / F.col("total_employees")) * 100, 2)
).orderBy(F.desc("attrition_rate"))

print("ATTRITION BY BUSINESS UNIT")
print("="*60)
display(bu_attrition)

## **4.5 Tenure (a long-serving employee with high job security) Analysis**

In [0]:
# Tenure statistics by employment status
tenure_stats = employee_enriched.groupBy("is_active").agg(
    F.avg("tenure_years").alias("avg_tenure_years"),
    F.min("tenure_years").alias("min_tenure_years"),
    F.max("tenure_years").alias("max_tenure_years"),
    F.expr("percentile(tenure_years, 0.5)").alias("median_tenure_years")
).withColumn(
    "status",
    F.when(F.col("is_active") == 1, "Active").otherwise("Departed")
)

print("TENURE COMPARISON: ACTIVE vs DEPARTED")
print("="*60)
display(tenure_stats.select("status", "avg_tenure_years", "median_tenure_years", "min_tenure_years", "max_tenure_years"))


# Tenure distribution for departed employees
print("TENURE AT EXIT - DISTRIBUTION")
print("="*60)

departed_tenure = employee_enriched.filter(F.col("is_active") == 0).select("tenure_years").toPandas()

if len(departed_tenure) > 0:
    plt.figure(figsize=(12, 6))
    plt.hist(departed_tenure['tenure_years'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    plt.axvline(departed_tenure['tenure_years'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {departed_tenure["tenure_years"].mean():.1f} years')
    plt.axvline(departed_tenure['tenure_years'].median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {departed_tenure["tenure_years"].median():.1f} years')
    plt.xlabel('Tenure at Exit (Years)', fontsize=12)
    plt.ylabel('Number of Employees', fontsize=12)
    plt.title('Distribution of Tenure at Exit', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## **4.6 Attrition Trends Over Time**

In [0]:
# Monthly attrition trend
monthly_exits = employee_enriched.filter(F.col("is_active") == 0).groupBy("exit_year", "exit_month").agg(
    F.count("*").alias("departures")
).orderBy("exit_year", "exit_month")

print("MONTHLY ATTRITION TREND")
print("="*60)
display(monthly_exits)


# Visualize monthly trend
monthly_exits_pd = monthly_exits.toPandas()

if len(monthly_exits_pd) > 0:
    monthly_exits_pd['year_month'] = monthly_exits_pd['exit_year'].astype(str) + '-' + monthly_exits_pd['exit_month'].astype(str).str.zfill(2)
    
    plt.figure(figsize=(14, 6))
    plt.plot(monthly_exits_pd['year_month'], monthly_exits_pd['departures'], marker='o', linewidth=2, markersize=6, color='darkred')
    plt.xlabel('Year-Month', fontsize=12)
    plt.ylabel('Number of Departures', fontsize=12)
    plt.title('Monthly Attrition Trend', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## **4.7 Termination Type Analysis**

In [0]:
termination_analysis = employee_enriched.filter(F.col("is_active") == 0).groupBy("TerminationType").agg(
    F.count("*").alias("count")
).withColumn(
    "percentage",
    F.round((F.col("count") / F.sum("count").over(Window.partitionBy())) * 100, 2)
).orderBy(F.desc("count"))

print("TERMINATION TYPE BREAKDOWN")
print("="*60)
display(termination_analysis)


# Top termination descriptions
termination_desc = employee_enriched.filter(F.col("is_active") == 0).groupBy("TerminationDescription").agg(
    F.count("*").alias("count")
).orderBy(F.desc("count")).limit(10)

print("TOP 10 TERMINATION REASONS")
print("="*60)
display(termination_desc)

## **4.8 Performance Score Analysis**

In [0]:
# Performance distribution for active vs departed
performance_comparison = employee_enriched.groupBy("is_active", "Performance Score").agg(
    F.count("*").alias("count")
).withColumn(
    "status",
    F.when(F.col("is_active") == 1, "Active").otherwise("Departed")
).orderBy("is_active", "Performance Score")

print("PERFORMANCE SCORE DISTRIBUTION")
print("="*60)
display(performance_comparison)

# Attrition rate by performance score
perf_attrition = employee_enriched.groupBy("Performance Score").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)).alias("departed")
).withColumn(
    "attrition_rate",
    F.round((F.col("departed") / F.col("total")) * 100, 2)
).orderBy("Performance Score")

print("ATTRITION RATE BY PERFORMANCE SCORE")
print("="*60)
display(perf_attrition)

## **4.9 Demographics Analysis**

In [0]:
# Age group attrition
age_attrition = employee_enriched.groupBy("age_group").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)).alias("departed")
).withColumn(
    "attrition_rate",
    F.round((F.col("departed") / F.col("total")) * 100, 2)
).orderBy("age_group")

print("ATTRITION BY AGE GROUP")
print("="*60)
display(age_attrition)


# Gender attrition
gender_attrition = employee_enriched.groupBy("GenderCode").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)).alias("departed")
).withColumn(
    "attrition_rate",
    F.round((F.col("departed") / F.col("total")) * 100, 2)
).orderBy(F.desc("total"))

print("ATTRITION BY GENDER")
print("="*60)
display(gender_attrition)


# Race/Ethnicity attrition
race_attrition = employee_enriched.groupBy("RaceDesc").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)).alias("departed")
).withColumn(
    "attrition_rate",
    F.round((F.col("departed") / F.col("total")) * 100, 2)
).orderBy(F.desc("total"))

print("ATTRITION BY RACE/ETHNICITY")
print("="*60)
display(race_attrition)


# Marital status attrition
marital_attrition = employee_enriched.groupBy("MaritalDesc").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)).alias("departed")
).withColumn(
    "attrition_rate",
    F.round((F.col("departed") / F.col("total")) * 100, 2)
).orderBy(F.desc("total"))

print("ATTRITION BY MARITAL STATUS")
print("="*60)
display(marital_attrition)

## **4.10 High-Risk Supervisor Analysis**

In [0]:
# Identify supervisors with highest attrition
supervisor_attrition = employee_enriched.groupBy("Supervisor").agg(
    F.count("*").alias("total_reports"),
    F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)).alias("departed")
).filter(F.col("total_reports") >= 5).withColumn(
    "attrition_rate",
    F.round((F.col("departed") / F.col("total_reports")) * 100, 2)
).orderBy(F.desc("attrition_rate")).limit(20)

print("TOP 20 SUPERVISORS WITH HIGHEST ATTRITION (min 5 reports)")
print("="*60)
display(supervisor_attrition)

# **5. Engagement Survey Analysis**

## **5.1 Engagement Score Distributions**

In [0]:
engagement_stats = engagement_df.agg(
    F.avg("Engagement Score").alias("avg_engagement"),
    F.avg("Satisfaction Score").alias("avg_satisfaction"),
    F.avg("Work-Life Balance Score").alias("avg_worklife"),
    F.expr("percentile(`Engagement Score`, 0.5)").alias("median_engagement"),
    F.expr("percentile(`Satisfaction Score`, 0.5)").alias("median_satisfaction"),
    F.expr("percentile(`Work-Life Balance Score`, 0.5)").alias("median_worklife")
)

print("ENGAGEMENT SURVEY SUMMARY STATISTICS")
print("="*60)
display(engagement_stats)


# Distribution of scores
engagement_pd = engagement_df.select(
    F.col("Engagement Score").alias("Engagement"),
    F.col("Satisfaction Score").alias("Satisfaction"),
    F.col("Work-Life Balance Score").alias("Work_Life_Balance")
).toPandas()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(engagement_pd['Engagement'].dropna(), bins=10, color='skyblue', edgecolor='black')
axes[0].set_title('Engagement Score Distribution')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Frequency')

axes[1].hist(engagement_pd['Satisfaction'].dropna(), bins=10, color='lightgreen', edgecolor='black')
axes[1].set_title('Satisfaction Score Distribution')
axes[1].set_xlabel('Score')

axes[2].hist(engagement_pd['Work_Life_Balance'].dropna(), bins=10, color='salmon', edgecolor='black')
axes[2].set_title('Work-Life Balance Score Distribution')
axes[2].set_xlabel('Score')

plt.tight_layout()
plt.show()

## **5.2 Correlation Between Engagement Metrics**

In [0]:
# Calculate correlation matrix
correlation_matrix = engagement_pd.corr()

print("CORRELATION MATRIX - ENGAGEMENT METRICS")
print("="*60)
print(correlation_matrix)

# Visualize correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap - Engagement Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## **5.3 Engagement vs Attrition (Join Analysis)**

In [0]:
# Join engagement data with employee data
engagement_attrition = engagement_df.alias("eng").join(
    employee_enriched.alias("emp"),
    F.col("eng.Employee ID") == F.col("emp.EmpID"),
    "inner"
).select(
    F.col("emp.EmpID"),
    F.col("emp.is_active"),
    F.col("eng.Engagement Score"),
    F.col("eng.Satisfaction Score"),
    F.col("eng.Work-Life Balance Score"),
    F.col("emp.DepartmentType"),
    F.col("emp.Performance Score")
)

# Compare engagement scores for active vs departed
engagement_comparison = engagement_attrition.groupBy("is_active").agg(
    F.avg("Engagement Score").alias("avg_engagement"),
    F.avg("Satisfaction Score").alias("avg_satisfaction"),
    F.avg("Work-Life Balance Score").alias("avg_worklife"),
    F.count("*").alias("count")
).withColumn(
    "status",
    F.when(F.col("is_active") == 1, "Active").otherwise("Departed")
)

print("ENGAGEMENT SCORES: ACTIVE vs DEPARTED EMPLOYEES")
print("="*60)
display(engagement_comparison)


# Low engagement employees - at risk analysis
low_engagement_threshold = 3  # Adjust based on your scale

at_risk_employees = engagement_attrition.filter(
    (F.col("Engagement Score") <= low_engagement_threshold) |
    (F.col("Satisfaction Score") <= low_engagement_threshold) |
    (F.col("Work-Life Balance Score") <= low_engagement_threshold)
).filter(F.col("is_active") == 1)

at_risk_count = at_risk_employees.count()
total_active_with_survey = engagement_attrition.filter(F.col("is_active") == 1).count()

print(f"AT-RISK EMPLOYEES (Low Engagement)")
print("="*60)
print(f"Employees with low scores: {at_risk_count:,}")
print(f"Total active with surveys: {total_active_with_survey:,}")
if total_active_with_survey > 0:
    print(f"At-risk percentage: {(at_risk_count/total_active_with_survey)*100:.2f}%")
print("\nSample at-risk employees:")
display(at_risk_employees.select("EmpID", "Engagement Score", "Satisfaction Score", "Work-Life Balance Score", "DepartmentType").limit(20))


# **6. Recruitment Data Analysis**

## **6.1 Hiring Pipeline Overview**

In [0]:
# Application status breakdown
status_breakdown = recruitment_df.groupBy("Status").agg(
    F.count("*").alias("count")
).withColumn(
    "percentage",
    F.round((F.col("count") / F.sum("count").over(Window.partitionBy())) * 100, 2)
).orderBy(F.desc("count"))

print("APPLICATION STATUS BREAKDOWN")
print("="*60)
display(status_breakdown)


# Visualize hiring funnel
status_pd = status_breakdown.toPandas().sort_values('count', ascending=False)

plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(range(len(status_pd)))
plt.bar(status_pd['Status'], status_pd['count'], color=colors, edgecolor='black')
plt.xlabel('Application Status', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)
plt.title('Hiring Pipeline - Application Status', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## **6.2 Applications by Job Title**

In [0]:

# Top job titles by application volume
job_title_apps = recruitment_df.groupBy("Job Title").agg(
    F.count("*").alias("total_applications"),
    F.sum(F.when(F.col("Status") == "Hired", 1).otherwise(0)).alias("hired"),
    F.sum(F.when(F.col("Status") == "Rejected", 1).otherwise(0)).alias("rejected")
).withColumn(
    "hire_rate",
    F.round((F.col("hired") / F.col("total_applications")) * 100, 2)
).orderBy(F.desc("total_applications")).limit(15)

print("TOP 15 JOB TITLES BY APPLICATION VOLUME")
print("="*60)
display(job_title_apps)


## **6.3 Candidate Demographics**

In [0]:
# Education level distribution
education_dist = recruitment_df.groupBy("Education Level").agg(
    F.count("*").alias("count")
).withColumn(
    "percentage",
    F.round((F.col("count") / F.sum("count").over(Window.partitionBy())) * 100, 2)
).orderBy(F.desc("count"))

print("EDUCATION LEVEL DISTRIBUTION")
print("="*60)
display(education_dist)

# Gender distribution of applicants
gender_dist = recruitment_df.groupBy("Gender").agg(
    F.count("*").alias("count"),
    F.sum(F.when(F.col("Status") == "Hired", 1).otherwise(0)).alias("hired")
).withColumn(
    "percentage",
    F.round((F.col("count") / F.sum("count").over(Window.partitionBy())) * 100, 2)
).withColumn(
    "hire_rate",
    F.round((F.col("hired") / F.col("count")) * 100, 2)
).orderBy(F.desc("count"))

print("GENDER DISTRIBUTION OF APPLICANTS")
print("="*60)
display(gender_dist)


# Experience distribution
experience_stats = recruitment_df.agg(
    F.avg("Years of Experience").alias("avg_experience"),
    F.expr("percentile(`Years of Experience`, 0.5)").alias("median_experience"),
    F.min("Years of Experience").alias("min_experience"),
    F.max("Years of Experience").alias("max_experience")
)

print("CANDIDATE EXPERIENCE STATISTICS")
print("="*60)
display(experience_stats)


# Experience distribution histogram
experience_pd = recruitment_df.select("Years of Experience").toPandas()

plt.figure(figsize=(12, 6))
plt.hist(experience_pd['Years of Experience'].dropna(), bins=20, color='teal', edgecolor='black', alpha=0.7)
plt.axvline(experience_pd['Years of Experience'].mean(), color='red', linestyle='--', linewidth=2, 
            label=f'Mean: {experience_pd["Years of Experience"].mean():.1f} years')
plt.xlabel('Years of Experience', fontsize=12)
plt.ylabel('Number of Applicants', fontsize=12)
plt.title('Distribution of Candidate Experience', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## **6.4 Salary Analysis**

In [0]:
# Desired salary statistics
salary_stats = recruitment_df.agg(
    F.avg("Desired Salary").alias("avg_desired_salary"),
    F.expr("percentile(`Desired Salary`, 0.5)").alias("median_desired_salary"),
    F.min("Desired Salary").alias("min_desired_salary"),
    F.max("Desired Salary").alias("max_desired_salary")
)

print("DESIRED SALARY STATISTICS")
print("="*60)
display(salary_stats)


# Desired salary by education level
salary_by_education = recruitment_df.groupBy("Education Level").agg(
    F.avg("Desired Salary").alias("avg_desired_salary"),
    F.count("*").alias("applicant_count")
).orderBy(F.desc("avg_desired_salary"))

print("DESIRED SALARY BY EDUCATION LEVEL")
print("="*60)
display(salary_by_education)


# Desired salary by experience
salary_experience_pd = recruitment_df.select("Years of Experience", "Desired Salary").toPandas()

plt.figure(figsize=(12, 6))
plt.scatter(salary_experience_pd['Years of Experience'], salary_experience_pd['Desired Salary'], 
            alpha=0.5, color='purple', edgecolors='black', linewidths=0.5)
plt.xlabel('Years of Experience', fontsize=12)
plt.ylabel('Desired Salary ($)', fontsize=12)
plt.title('Desired Salary vs Years of Experience', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## **6.5 Geographic Analysis**

In [0]:
# Applications by state
state_apps = recruitment_df.groupBy("State").agg(
    F.count("*").alias("applications"),
    F.sum(F.when(F.col("Status") == "Hired", 1).otherwise(0)).alias("hired")
).withColumn(
    "hire_rate",
    F.round((F.col("hired") / F.col("applications")) * 100, 2)
).orderBy(F.desc("applications")).limit(15)

print("TOP 15 STATES BY APPLICATION VOLUME")
print("="*60)
display(state_apps)

# **7. Training & Development Analysis**

## **7.1 Training Program Overview**

In [0]:
# Training program participation
program_stats = training_df.groupBy("Training Program Name").agg(
    F.count("*").alias("participants"),
    F.avg("Training Duration(Days)").alias("avg_duration_days"),
    F.sum("Training Cost").alias("total_cost"),
    F.avg("Training Cost").alias("avg_cost_per_participant")
).orderBy(F.desc("participants")).limit(15)

print("TOP 15 TRAINING PROGRAMS BY PARTICIPATION")
print("="*60)
display(program_stats)


## **7.2 Training by Type**

In [0]:
# Training type breakdown
training_type_stats = training_df.groupBy("Training Type").agg(
    F.count("*").alias("sessions"),
    F.countDistinct("Employee ID").alias("unique_employees"),
    F.sum("Training Cost").alias("total_investment"),
    F.avg("Training Duration(Days)").alias("avg_duration")
).orderBy(F.desc("sessions"))

print("TRAINING BY TYPE")
print("="*60)
display(training_type_stats)


# Visualize training investment by type
training_type_pd = training_type_stats.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].bar(training_type_pd['Training Type'], training_type_pd['sessions'], 
            color='steelblue', edgecolor='black')
axes[0].set_xlabel('Training Type')
axes[0].set_ylabel('Number of Sessions')
axes[0].set_title('Training Sessions by Type')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(training_type_pd['Training Type'], training_type_pd['total_investment'], 
            color='green', edgecolor='black')
axes[1].set_xlabel('Training Type')
axes[1].set_ylabel('Total Investment ($)')
axes[1].set_title('Training Investment by Type')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## **7.3 Training Outcomes**

In [0]:
# Training outcome distribution
outcome_stats = training_df.groupBy("Training Outcome").agg(
    F.count("*").alias("count")
).withColumn(
    "percentage",
    F.round((F.col("count") / F.sum("count").over(Window.partitionBy())) * 100, 2)
).orderBy(F.desc("count"))

print("TRAINING OUTCOME DISTRIBUTION")
print("="*60)
display(outcome_stats)


## **7.4 Training Cost Analysis**

In [0]:
# Total training investment
total_training_investment = training_df.agg(F.sum("Training Cost")).collect()[0][0]
total_training_participants = training_df.select("Employee ID").distinct().count()
avg_cost_per_employee = total_training_investment / total_training_participants if total_training_participants > 0 else 0

print("TRAINING INVESTMENT SUMMARY")
print("="*60)
print(f"Total Training Investment: ${total_training_investment:,.2f}")
print(f"Unique Employees Trained: {total_training_participants:,}")
print(f"Avg Cost per Employee: ${avg_cost_per_employee:,.2f}")


# Training duration distribution
duration_pd = training_df.select("Training Duration(Days)").toPandas()

plt.figure(figsize=(12, 6))
plt.hist(duration_pd['Training Duration(Days)'].dropna(), bins=20, color='orange', edgecolor='black', alpha=0.7)
plt.xlabel('Training Duration (Days)', fontsize=12)
plt.ylabel('Number of Training Sessions', fontsize=12)
plt.title('Distribution of Training Duration', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## **7.5 Top Trainers**

In [0]:
# Trainer effectiveness
trainer_stats = training_df.groupBy("Trainer").agg(
    F.count("*").alias("sessions_conducted"),
    F.countDistinct("Employee ID").alias("employees_trained"),
    F.sum("Training Cost").alias("total_program_cost")
).orderBy(F.desc("sessions_conducted")).limit(10)

print("TOP 10 TRAINERS BY SESSION COUNT")
print("="*60)
display(trainer_stats)


# **8. Cross-Table Analysis (Most Valuable Insights!)**

## **8.1 New Hire Retention Analysis**

In [0]:
# Match recruitment data to employee data (hired candidates only)
hired_candidates = recruitment_df.filter(F.col("Status") == "Hired").select(
    F.col("First Name").alias("rec_first"),
    F.col("Last Name").alias("rec_last"),
    F.col("Email").alias("rec_email"),
    F.col("Education Level"),
    F.col("Years of Experience"),
    F.col("Desired Salary"),
    F.col("Job Title").alias("applied_for_title")
)

# Join with employee data on name match (simplified - in production use better matching)
new_hire_analysis = employee_enriched.join(
    hired_candidates,
    (F.col("FirstName") == F.col("rec_first")) & (F.col("LastName") == F.col("rec_last")),
    "inner"
).select(
    F.col("EmpID"),
    F.col("FirstName"),
    F.col("LastName"),
    F.col("is_active"),
    F.col("tenure_years"),
    F.col("tenure_months"),
    F.col("Education Level"),
    F.col("Years of Experience"),
    F.col("DepartmentType"),
    F.col("Performance Score")
).withColumn(
    "first_year_departure",
    F.when((F.col("is_active") == 0) & (F.col("tenure_years") < 1), 1).otherwise(0)
).withColumn(
    "ninety_day_departure",
    F.when((F.col("is_active") == 0) & (F.col("tenure_months") <= 3), 1).otherwise(0)
)

# Calculate new hire retention metrics
total_new_hires = new_hire_analysis.count()

print("NEW HIRE RETENTION METRICS")
print("="*60)

if total_new_hires > 0:
    first_year_departures = new_hire_analysis.filter(F.col("first_year_departure") == 1).count()
    ninety_day_departures = new_hire_analysis.filter(F.col("ninety_day_departure") == 1).count()
    
    print(f"Total New Hires Matched: {total_new_hires:,}")
    print(f"90-Day Departures: {ninety_day_departures:,} ({(ninety_day_departures/total_new_hires)*100:.2f}%)")
    print(f"First-Year Departures: {first_year_departures:,} ({(first_year_departures/total_new_hires)*100:.2f}%)")
    
    # New hire attrition by education level
    new_hire_by_education = new_hire_analysis.groupBy("Education Level").agg(
        F.count("*").alias("total_hires"),
        F.sum("first_year_departure").alias("first_year_exits")
    ).withColumn(
        "first_year_attrition_rate",
        F.round((F.col("first_year_exits") / F.col("total_hires")) * 100, 2)
    ).orderBy(F.desc("first_year_attrition_rate"))
    
    print("\nNEW HIRE ATTRITION BY EDUCATION LEVEL")
    print("="*60)
    display(new_hire_by_education)
else:
    print(f"Total New Hires Matched: {total_new_hires:,}")
    print("\n⚠️  No matches found between recruitment and employee data.")
    print("This could be due to:")
    print("  • Different name formats (e.g., 'John' vs 'Jonathan')")
    print("  • Timing mismatch (hired but not yet in employee_data)")
    print("  • Different data sources or missing ADEmail field")
    print("\n📊 Alternative Analysis: Analyzing all new hires based on tenure...")
    
    # Alternative: Analyze employees with less than 2 years tenure as "new hires"
    new_hires_by_tenure = employee_enriched.filter(F.col("tenure_years") <= 2).withColumn(
        "first_year_departure",
        F.when((F.col("is_active") == 0) & (F.col("tenure_years") < 1), 1).otherwise(0)
    ).withColumn(
        "ninety_day_departure",
        F.when((F.col("is_active") == 0) & (F.col("tenure_months") <= 3), 1).otherwise(0)
    )
    
    total_new_by_tenure = new_hires_by_tenure.count()
    
    if total_new_by_tenure > 0:
        first_year_exits = new_hires_by_tenure.filter(F.col("first_year_departure") == 1).count()
        ninety_day_exits = new_hires_by_tenure.filter(F.col("ninety_day_departure") == 1).count()
        
        print(f"\n✅ Employees with ≤2 years tenure: {total_new_by_tenure:,}")
        print(f"   90-Day Departures: {ninety_day_exits:,} ({(ninety_day_exits/total_new_by_tenure)*100:.2f}%)")
        print(f"   First-Year Departures: {first_year_exits:,} ({(first_year_exits/total_new_by_tenure)*100:.2f}%)")
        
        # Show tenure-based new hire analysis
        tenure_new_hire_analysis = new_hires_by_tenure.groupBy("DepartmentType").agg(
            F.count("*").alias("new_hires"),
            F.sum("first_year_departure").alias("exits")
        ).withColumn(
            "attrition_rate",
            F.round((F.col("exits") / F.col("new_hires")) * 100, 2)
        ).orderBy(F.desc("attrition_rate"))
        
        print("\n   New Hire Attrition by Department (tenure ≤2 years):")
        display(tenure_new_hire_analysis)

## **8.2 Training Impact on Retention**

In [0]:
# Join training data with employee data
training_retention = training_df.join(
    employee_enriched,
    training_df["Employee ID"] == employee_enriched["EmpID"],
    "inner"
).select(
    F.col("EmpID"),
    F.col("is_active"),
    F.col("Training Program Name"),
    F.col("Training Type"),
    F.col("Training Outcome"),
    F.col("Training Cost"),
    F.col("Performance Score"),
    F.col("DepartmentType")
)

# Compare attrition: trained vs untrained employees
employees_with_training = training_retention.select("EmpID").distinct()
total_trained = employees_with_training.count()
trained_departed = training_retention.filter(F.col("is_active") == 0).select("EmpID").distinct().count()

untrained_employees = employee_enriched.join(employees_with_training, "EmpID", "left_anti")
total_untrained = untrained_employees.count()
untrained_departed = untrained_employees.filter(F.col("is_active") == 0).count()

print("TRAINING IMPACT ON RETENTION")
print("="*60)

if total_trained > 0:
    print(f"Employees with Training: {total_trained:,}")
    print(f"Attrition Rate (Trained): {(trained_departed/total_trained)*100:.2f}%")
else:
    print(f"Employees with Training: {total_trained:,}")
    print("⚠️ No employees with training records found")

if total_untrained > 0:
    print(f"\nEmployees without Training: {total_untrained:,}")
    print(f"Attrition Rate (Untrained): {(untrained_departed/total_untrained)*100:.2f}%")
else:
    print(f"\nEmployees without Training: {total_untrained:,}")
    print("✅ All employees have training records!")

if total_trained > 0 and total_untrained > 0:
    trained_rate = (trained_departed/total_trained)*100
    untrained_rate = (untrained_departed/total_untrained)*100
    print(f"\nDifference: {abs(trained_rate - untrained_rate):.2f} percentage points")
    if trained_rate < untrained_rate:
        print(f"✅ Training reduces attrition by {untrained_rate - trained_rate:.2f} percentage points")
    else:
        print(f"⚠️ Trained employees have higher attrition by {trained_rate - untrained_rate:.2f} percentage points")


# Training outcome vs attrition
training_outcome_retention = training_retention.groupBy("Training Outcome").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)).alias("departed")
).withColumn(
    "attrition_rate",
    F.round((F.col("departed") / F.col("total")) * 100, 2)
).orderBy(F.desc("attrition_rate"))

print("ATTRITION BY TRAINING OUTCOME")
print("="*60)
display(training_outcome_retention)


## **8.3 Training & Performance Correlation**

In [0]:

# Average performance score by training status
training_performance = training_retention.groupBy("Training Outcome", "Performance Score").agg(
    F.count("*").alias("count")
).orderBy("Training Outcome", "Performance Score")

print("TRAINING OUTCOME vs PERFORMANCE SCORE")
print("="*60)

performance_count = training_performance.count()
if performance_count > 0:
    display(training_performance)
else:
    print("⚠️ No performance data available for trained employees")


## **8.4 Engagement, Training & Attrition Triangle**

In [0]:
# Join all three: engagement + training + employee data
comprehensive_analysis = engagement_attrition.join(
    training_retention.select("EmpID", "Training Outcome").distinct(),
    "EmpID",
    "left"
).withColumn(
    "has_training",
    F.when(F.col("Training Outcome").isNotNull(), 1).otherwise(0)
)

# Segment analysis
engagement_training_segments = comprehensive_analysis.groupBy("has_training", "is_active").agg(
    F.avg("Engagement Score").alias("avg_engagement"),
    F.avg("Satisfaction Score").alias("avg_satisfaction"),
    F.count("*").alias("count")
).withColumn(
    "training_status",
    F.when(F.col("has_training") == 1, "Trained").otherwise("Not Trained")
).withColumn(
    "employment_status",
    F.when(F.col("is_active") == 1, "Active").otherwise("Departed")
)

print("ENGAGEMENT SCORES: TRAINED vs UNTRAINED × ACTIVE vs DEPARTED")
print("="*60)

segment_count = engagement_training_segments.count()
if segment_count > 0:
    display(engagement_training_segments.select("training_status", "employment_status", "avg_engagement", "avg_satisfaction", "count"))
else:
    print("⚠️ Insufficient data for cross-tabulation analysis")

# **9. Data Quality Assessment**

## **9.1 Duplicate Records Check**

In [0]:
# Check for duplicate EmpIDs
duplicate_ids = employee_df.groupBy("EmpID").agg(
    F.count("*").alias("count")
).filter(F.col("count") > 1)

dup_count = duplicate_ids.count()
print(f"Duplicate EmpIDs found: {dup_count}")

if dup_count > 0:
    display(duplicate_ids)

## **9.2 Invalid Date Records**

In [0]:
# Check for exit dates before start dates
invalid_dates = employee_enriched.filter(
    (F.col("ExitDate_parsed").isNotNull()) & 
    (F.col("ExitDate_parsed") < F.col("StartDate_parsed"))
).select("EmpID", "StartDate", "ExitDate", "StartDate_parsed", "ExitDate_parsed")

invalid_count = invalid_dates.count()
print(f"Records with ExitDate before StartDate: {invalid_count}")

if invalid_count > 0:
    display(invalid_dates)

## **9.3 Status Mismatches**

In [0]:
# Employees marked as Active but have an ExitDate
status_mismatch = employee_enriched.filter(
    (F.col("EmployeeStatus") == "Active") & 
    (F.col("ExitDate_parsed").isNotNull())
).select("EmpID", "FirstName", "LastName", "EmployeeStatus", "ExitDate", "is_active")

mismatch_count = status_mismatch.count()
print(f"Active employees with ExitDate: {mismatch_count}")

if mismatch_count > 0:
    display(status_mismatch.limit(20))

# **10. Feature Engineering for ML Model**

## **10.1 Create Comprehensive Feature Set**

In [0]:
# Department-level features
dept_features = employee_enriched.groupBy("DepartmentType").agg(
    F.avg("tenure_months").alias("dept_avg_tenure_months"),
    (F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)) / F.count("*")).alias("dept_attrition_rate"),
    F.count("*").alias("dept_size")
)

# Supervisor-level features
supervisor_features = employee_enriched.groupBy("Supervisor").agg(
    F.count("*").alias("supervisor_team_size"),
    (F.sum(F.when(F.col("is_active") == 0, 1).otherwise(0)) / F.count("*")).alias("supervisor_attrition_rate")
)

# Training features
training_features = training_df.groupBy("Employee ID").agg(
    F.count("*").alias("total_training_sessions"),
    F.sum("Training Cost").alias("total_training_investment"),
    F.sum("Training Duration(Days)").alias("total_training_days"),
    F.collect_list("Training Outcome").alias("training_outcomes")
).withColumn(
    "has_successful_training",
    F.when(F.array_contains(F.col("training_outcomes"), "Completed"), 1).otherwise(0)
)

# Join all features together
ml_feature_set = employee_enriched.join(
    dept_features, "DepartmentType", "left"
).join(
    supervisor_features, "Supervisor", "left"
).join(
    engagement_df.withColumnRenamed("Employee ID", "EmpID"),
    "EmpID", "left"
).join(
    training_features.withColumnRenamed("Employee ID", "EmpID"),
    "EmpID", "left"
).select(
    # Target variable
    "EmpID",
    "is_active",
    
    # Tenure features
    "tenure_months",
    "tenure_years",
    
    # Demographics
    "age",
    "age_group",
    "GenderCode",
    "MaritalDesc",
    "RaceDesc",
    
    # Job features
    "DepartmentType",
    "BusinessUnit",
    "Title",
    "PayZone",
    "EmployeeType",
    
    # Performance
    "Performance Score",
    "Current Employee Rating",
    
    # Engagement
    "Engagement Score",
    "Satisfaction Score",
    "Work-Life Balance Score",
    
    # Training
    "total_training_sessions",
    "total_training_investment",
    "has_successful_training",
    
    # Aggregated features
    "dept_avg_tenure_months",
    "dept_attrition_rate",
    "dept_size",
    "supervisor_team_size",
    "supervisor_attrition_rate"
)

print("ML FEATURE SET PREVIEW")
print("="*60)
display(ml_feature_set.limit(10))

print(f"\nTotal Features: {len(ml_feature_set.columns)}")
print(f"Total Records: {ml_feature_set.count():,}")


# **11. Key Insights Summary**

In [0]:
print("="*80)
print("EDA - KEY INSIGHTS SUMMARY")
print("="*80)

print("\n📊 DATA OVERVIEW:")
print(f"   • Total Employees: {total_employees:,}")
print(f"   • Active: {active_employees:,} | Departed: {departed_employees:,}")
if total_employees > 0:
    print(f"   • Overall Attrition Rate: {attrition_rate:.2f}%")

print("\n🎯 TOP FINDINGS:")

# Top attrition department
top_dept = dept_attrition.first()
if top_dept:
    print(f"   • Highest Attrition Department: {top_dept['DepartmentType']} ({top_dept['attrition_rate']:.1f}%)")

# Average tenure comparison
if active_employees > 0:
    active_avg_tenure = employee_enriched.filter(F.col("is_active") == 1).agg(F.avg("tenure_years")).collect()[0][0]
    if active_avg_tenure:
        print(f"   • Avg Tenure - Active: {active_avg_tenure:.1f} years", end="")
    
if departed_employees > 0:
    departed_avg_tenure = employee_enriched.filter(F.col("is_active") == 0).agg(F.avg("tenure_years")).collect()[0][0]
    if departed_avg_tenure:
        if active_employees > 0:
            print(f" | Departed: {departed_avg_tenure:.1f} years")
        else:
            print(f"   • Avg Tenure - Departed: {departed_avg_tenure:.1f} years")

# Voluntary vs involuntary
if departed_employees > 0:
    print(f"   • Voluntary Departures: {(voluntary_count/departed_employees)*100:.1f}%")

print("\n💼 RECRUITMENT INSIGHTS:")
total_applicants = recruitment_df.count()
hired_count = recruitment_df.filter(F.col("Status") == "Hired").count()
print(f"   • Total Applications: {total_applicants:,}")
if total_applicants > 0:
    print(f"   • Hired: {hired_count:,} ({(hired_count/total_applicants)*100:.1f}%)")
else:
    print(f"   • Hired: {hired_count:,}")

print("\n📚 TRAINING INSIGHTS:")
if total_training_investment:
    print(f"   • Total Training Investment: ${total_training_investment:,.2f}")
if total_training_participants > 0:
    print(f"   • Employees Trained: {total_training_participants:,}")
if total_trained > 0 and total_untrained > 0:
    print(f"   • Attrition (Trained): {(trained_departed/total_trained)*100:.1f}% vs (Untrained): {(untrained_departed/total_untrained)*100:.1f}%")
elif total_trained > 0:
    print(f"   • Attrition (Trained): {(trained_departed/total_trained)*100:.1f}%")

print("\n⚠️  DATA QUALITY ISSUES:")
print(f"   • Duplicate EmpIDs: {dup_count}")
print(f"   • Invalid Dates: {invalid_count}")
print(f"   • Status Mismatches: {mismatch_count}")
